## Building a GPT - DORA ALKAN

6 haftadır beklediğimiz o gün geldi. 🙏🏻

In [1]:
# Task-1: Tiny Shakespeare'i oku, karakter tokenizer'ını yaz (encode, decode), veriyi %90 train, %10 val diye ayır. Rastgele parçalardan batch çeken get_batch fonksiyonunu yaz. Hafta 2'deki bigram modelini nn.Module olarak kur, eğit, val loss'u kaydet. Bu senin karşılaştırma tabanın. (7:52 - 38:00)

In [2]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
    'input.txt'
)

('input.txt', <http.client.HTTPMessage at 0x107e5a0d0>)

In [3]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
# let's look at the first 1000 characters
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
#When people say tokenize, they mean convert the raw text as a string to some sequence of integers according to some vocabulary of possible elements.

In [30]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [9]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [10]:
n = int(0.9*len(data)) # datanın yuzde doksani, train set oalcak bu.
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8
# len(train_data[block_size:]) #bu iki noktaya göre neresi boşsa orayı sınırlamamış oluyoruz. ona göre düşün bunun datayı nasıl kestiğini
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"If input is {context}, output is {target}")

If input is tensor([18]), output is 47
If input is tensor([18, 47]), output is 56
If input is tensor([18, 47, 56]), output is 57
If input is tensor([18, 47, 56, 57]), output is 58
If input is tensor([18, 47, 56, 57, 58]), output is 1
If input is tensor([18, 47, 56, 57, 58,  1]), output is 15
If input is tensor([18, 47, 56, 57, 58,  1, 15]), output is 47
If input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), output is 58


In [22]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    #lets generate a batch of inputs x and targets y
    data = train_data if split == 'train' else val_data
    # Yukarıdaki expression ternary, Uzun ve tanıdık hali aşağıda:
    # if split == 'train':
    #     data = train_data
    # else:
    #     data = val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) 
    #torch.randint --> Returns a tensor filled with random integers generated uniformly between low (inclusive) and high (exclusive).
    #torch.randint(low, high, size): Burada ilk argüman gerçekten alt sınır, ikinci üst sınır.
    #torch.randint(high, size): Burada tek argüman varsa, PyTorch bunu üst sınır olarak yorumluyor, alt sınırı otomatik 0 yapıyor. VIDEODAKI KULLANIM BU.
    #len(data) - block_size: high sınır data dışa taşmasın diye böyle sçildi.
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) #bütün sekans bi sağa kayıyor x te olan
    #torch.stack(..) --> tek boyutlu bu tensorları ust uste koyuyor ehr biri bir satır olacak biçimde.
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)


print('-----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"If input is {context.tolist()}, output is {target}")
    

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
If input is [24], output is 43
If input is [24, 43], output is 58
If input is [24, 43, 58], output is 5
If input is [24, 43, 58, 5], output is 57
If input is [24, 43, 58, 5, 57], output is 1
If input is [24, 43, 58, 5, 57, 1], output is 46
If input is [24, 43, 58, 5, 57, 1, 46], output is 43
If input is [24, 43, 58, 5, 57, 1, 46, 43], output is 39
If input is [44], output is 53
If input is [44, 53], output is 56
If input is [44, 53, 56], output is 1
If input is [44, 53, 56, 1], output is 58
If input is [44, 53, 56, 1, 58], output is 46
If input is [44, 53, 56, 1, 58, 46], output is 39
If input i

In [ ]:
# #Farkı MLP'den (hafta 4-6) — işte burada asıl yenilik

# MLP'de (WaveNet dahil): bir forward pass'te, model sadece 1 tahmin üretiyordu — "8 karakterin tamamını gördüm, 9.yu tahmin et." Eğer 7 karakter sonrası tahmin etmek istiyorsan, ayrı bir forward pass gerekiyordu.

# Transformer'da: tek bir forward pass'te, model 8 tahminin hepsini aynı anda üretebiliyor — "1 karakterden sonrasını da, 2 karakterden sonrasını da, ..., 8 karakterden sonrasını da, hepsini bir kerede söyle."

In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): #subclass of nn Module

    def __init__(self, vocab_size): #__init__ — model kurulurken ne yapılacak
        super().__init__() #bu satır zorunlu, her zaman ilk satır olarak yazılır. "Önce, miras aldığım nn.Module'ün kendi kurulumunu çalıştır" demek. Bunu atlarsan, PyTorch'un iç mekanizması (parametre takibi gibi) bozuk çalışır.
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None): #idx --> girdi, bizim xb gibi bir tensor ([batch, block_size] şeklinde, harf indeksleri).

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(B,T,C) where C = vocab_size, bizim batchdeki her bi harf indexi bu random embedding tensorunun o sayıncı rowunu alıyor. Hep öyleydi zaten.

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)  #targets da (B,T) idi, onu da (B*T,) yapıyoruz ki loss functiona uygun olsun.

            loss = F.cross_entropy(logits, targets)  #(cross_entropy (B, C, T) bekliyor argumanı. sıra degisik.)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) 
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# Somut, gerçek xb'nle başlayalım
# python
# xb = [
#   [24, 43, 58, 5, 57, 1, 46, 43],   # dizi 0
#   [44, 53, 56, 1, 58, 46, 43, 56],  # dizi 1 (uydurma örnek)
#   [12, 8, 15, 23, 1, 9, 44, 5],     # dizi 2
#   [1, 5, 32, 44, 18, 23, 9, 1],     # dizi 3
# ]

# Şekli [4, 8] — 4 satır (dizi), her satırda 8 sayı (karakter indeksi).

# logits = embedding_table(xb) çağırınca ne olur

# Her tek sayı, yerini 65 sayılık bir skor listesine bırakıyor. Yani xb[0][0]=24 idi, şimdi logits[0][0] 65 tane sayı oluyor:

# python
# logits[0][0] = [0.3, -1.2, 0.8, 2.1, -0.4, ..., 0.1]   # 65 sayı, bu "24" indeksinin tablo satırı

In [35]:
#optimize
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    #sample a batch
    xb, yb = get_batch('train')

    #evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 100 == 0:
        print(steps, loss.item())

0 2.5639915466308594
100 2.4837474822998047
200 2.364332437515259
300 2.5103583335876465
400 2.4930427074432373
500 2.449208974838257
600 2.5211586952209473
700 2.4665935039520264
800 2.346386432647705
900 2.4973840713500977
1000 2.4571404457092285
1100 2.4357669353485107
1200 2.537367582321167
1300 2.4590628147125244
1400 2.4704790115356445
1500 2.48294997215271
1600 2.4412147998809814
1700 2.4459030628204346
1800 2.3872170448303223
1900 2.5263919830322266
2000 2.41656494140625
2100 2.4955949783325195
2200 2.405557155609131
2300 2.427614212036133
2400 2.503789186477661
2500 2.409576654434204
2600 2.441650867462158
2700 2.5540759563446045
2800 2.4418482780456543
2900 2.456915855407715
3000 2.491800308227539
3100 2.4716358184814453
3200 2.443476676940918
3300 2.4190330505371094
3400 2.4931533336639404
3500 2.386691093444824
3600 2.409395933151245
3700 2.57279634475708
3800 2.4571831226348877
3900 2.5732898712158203
4000 2.422809600830078
4100 2.533522844314575
4200 2.4947657585144043
43

In [55]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Yomirs
Ne be'HE:
So,
TER:
'st or, hat ouththif!
Reshariearcay'd olive, the BO:
Fle
' Ast f icoustyoy,
Byon d atholllowis BUSe, th e s th,
ats tald thisotoonthrthised t, d s fum, arerouen Math perauthe bery.
Wheron a,
Wh tor fid y hesthonemaie tigos, s d ysthil gousho; wof thior,
Dur ute me pre far thry epol myomy bunge ad G s ve.

S:
Thepan, ake my t Whe for more y t, t t baldersowifathth toucharto Hounst;

BE sin tim;
LENI is ds:
wor My buansarang med t at w inou ba this owisturg he ike wed, g 


In [46]:
# Task-2: Geçmişin ortalamasını üç yoldan al: for döngüsüyle, lower triangular matrisle (torch.tril) çarparak, softmax'la. Üçünün aynı sonucu verdiğini torch.allclose ile göster. Matris çarpımının neden ağırlıklı ortalama olduğunu kendi cümlelerinle yaz. (42:13 - 58:26)

In [ ]:
#Math trick

torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
#şöyle bir sistem istiyoruz. mesela 8 li veri içinde 4. sıradakini ele alalım. bu 4. sıradaki index kendinde önceki 1,2 ve 3. indexle haberleşebilsin.
#Haberleşmek dediğimiz de oradaki değerlerle bi ilişki içinde olmak. en basit yolu ortalama almak. ne alaka dersen mesela 100, 101, 102 ve 1, 2, 3 diye 2 örnek olsun.
#Bunların ortalaması bu sayıların buyuklukleri hakkında az çok fikir sahibi olmamızı saülıyor ama mesela 1, 400, 2 gibi de olabilird. bu tarz durumlar arasındaki farkı da kaybediyoruz. Böyle düşün.

In [ ]:
#1 - FOR DONGUSU
xbow = torch.zeros((B, T, C)) # (B, T, T) boyutunda sıfırlarla dolu tensor, bow = bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0) 

In [ ]:
#2 - MATRIS CARPIMI
wei = torch.tril(torch.ones(T, T)) # (T, T) lower triangular matrix
wei = wei / wei.sum(1, keepdim=True) # normalize so that each row sums to 1
xbow2 = wei @ x # (T, T) @ (B, T, C) --> (B, T, T) @ (B, T, C) --> (B, T, C) weighted average
torch.allclose(xbow, xbow2)

True

In [67]:
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [ ]:
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # fill the lower triangular part with -inf
# wei = F.softmax(wei, dim=-1) #softmaxte exponentiate ediyoz hepsini ve sum a bölüyoz. aynı şey oluyıo yani row toplamları 1 oluyor.
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
#3 - SOFTMAX
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # fill the lower triangular part with -inf
wei = F.softmax(wei, dim=-1)
xbow3= wei @ x
torch.allclose(xbow, xbow3)



True

In [51]:
torch.tril(torch.ones(3, 3)) #lower triangular matrix, alt üçgen matris. 

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [50]:
torch.manual_seed(42)
a = torch.ones(3, 3)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print('a=')
print(a)
print('----')
print('b=')
print(b)
print('----')
print('c=')
print(c)
print('----')

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
----
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])
----


In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True) #bütün rowları normalize ediyoruz ki ağırlıklı ortalama olsun. (row sum = 1)

# Neden "satır toplamı 1" demek "ağırlıklı ortalama" demek
# Ağırlıklı ortalamanın tanımı: bir grup sayıyı, her birine farklı bir "önem/ağırlık" vererek birleştirmek — ama ağırlıkların toplamı 1 olmak zorunda (yoksa sonuç anlamsız büyür/küçülür).
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print('a=')
print(a)
print('----')
print('b=')
print(b)
print('----')
print('c=')
print(c)
print('----')

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
----
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
----
